In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import gc

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
train_path = "/content/drive/MyDrive/MajorProject/CIC_IoT_2023/Full_Dataset/train_working.csv"
test_path  = "/content/drive/MyDrive/MajorProject/CIC_IoT_2023/Full_Dataset/test_frozen.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (900000, 47)
Test shape: (259646, 47)


/tmp/ipykernel_7438/1763718454.py:5: DtypeWarning: Columns (42) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df  = pd.read_csv(test_path)


### Data cleaning

In [4]:
print(f"Shape of training data: {train_df.shape}")
print(f"Shape of test data: {test_df.shape}")

Shape of training data: (900000, 47)
Shape of test data: (259646, 47)


In [5]:
LABEL_COL = 'label'

train_df = train_df.dropna(subset=[LABEL_COL])
test_df = test_df.dropna(subset=[LABEL_COL])

print("Train shape after cleaning", train_df.shape)
print("Test shape after cleaning", test_df.shape)

Train shape after cleaning (900000, 47)
Test shape after cleaning (259645, 47)


In [6]:
test_df["Radius"] = pd.to_numeric(test_df["Radius"], errors="coerce")

### Split features and target

In [7]:
X_train = train_df.drop(columns=[LABEL_COL])
y_train = train_df[LABEL_COL]

X_test = test_df.drop(columns=[LABEL_COL])
y_test = test_df[LABEL_COL]

### Label encoding

In [8]:
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print("Number of classes:", len(le.classes_))

Number of classes: 34


### Convert to LGBM Dataset

In [9]:
train_data = lgb.Dataset(X_train, label=y_train_enc)
test_data  = lgb.Dataset(X_test, label=y_test_enc)

### Define LightGBM

In [12]:
params = {
    "objective": "multiclass",
    "num_class": len(le.classes_),
    "metric": "multi_logloss",

    # better learning dynamics
    "learning_rate": 0.05,
    "num_leaves": 50,          # ↑ more capacity
    "max_depth": -1,           # allow deeper trees
    "min_data_in_leaf": 50,    # still controlled

    # regularization (VERY important)
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,

    # extra stability
    "lambda_l1": 0.5,
    "lambda_l2": 0.5,

    "verbosity": -1,
    "seed": 42
}

In [13]:
model = lgb.train(
    params,
    train_data,
    num_boost_round=100,   
    valid_sets=[test_data],
    valid_names=["test"],
    callbacks=[
        lgb.log_evaluation(20),
        lgb.early_stopping(50)
    ]
)

Training until validation scores don't improve for 50 rounds
[20]	test's multi_logloss: 0.51639
[40]	test's multi_logloss: 0.300163
[60]	test's multi_logloss: 0.253266
[80]	test's multi_logloss: 0.238628
[100]	test's multi_logloss: 0.231692
Did not meet early stopping. Best iteration is:
[100]	test's multi_logloss: 0.231692


In [14]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

In [15]:
print("Accuracy:", accuracy_score(y_test_enc, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))

Accuracy: 0.9176837605191704

Classification Report:

                         precision    recall  f1-score   support

       Backdoor_Malware       0.54      0.22      0.31       638
          BenignTraffic       0.75      0.75      0.75      6000
       BrowserHijacking       0.90      0.60      0.72      1167
       CommandInjection       0.56      0.48      0.51      1078
 DDoS-ACK_Fragmentation       1.00      1.00      1.00      7799
        DDoS-HTTP_Flood       1.00      0.99      1.00      5742
        DDoS-ICMP_Flood       1.00      1.00      1.00      6000
DDoS-ICMP_Fragmentation       1.00      1.00      1.00      7655
      DDoS-PSHACK_Flood       1.00      1.00      1.00      6000
       DDoS-RSTFINFlood       1.00      1.00      1.00      6000
         DDoS-SYN_Flood       1.00      1.00      1.00      6000
         DDoS-SlowLoris       1.00      1.00      1.00      4672
DDoS-SynonymousIP_Flood       1.00      1.00      1.00      6000
         DDoS-TCP_Flood       1.00 

In [16]:
import os
import pickle
import numpy as np

# 🔧 Change this path if needed
SAVE_DIR = "/content/drive/MyDrive/MajorProject/Baseline_LGBM"

os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save model
model.save_model(os.path.join(SAVE_DIR, "lgbm_model.txt"))

# 2. Save predictions
np.save(os.path.join(SAVE_DIR, "y_pred.npy"), y_pred)
np.save(os.path.join(SAVE_DIR, "y_pred_probs.npy"), y_pred_probs)

# 3. Save label encoder
with open(os.path.join(SAVE_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)

# 4. Save test data (for XAI)
X_test.to_csv(os.path.join(SAVE_DIR, "X_test.csv"), index=False)
y_test.to_csv(os.path.join(SAVE_DIR, "y_test.csv"), index=False)

# 5. Save feature names (optional but useful)
with open(os.path.join(SAVE_DIR, "feature_names.pkl"), "wb") as f:
    pickle.dump(X_train.columns.tolist(), f)

print(f"✅ All artifacts saved to: {SAVE_DIR}")

✅ All artifacts saved to: /content/drive/MyDrive/MajorProject/Baseline_LGBM
